## Testdaten
Für das Projekt müssen zumindest einmalig Testdaten generiert werden, die zum RAG-Datensatz passen. Das werde ich wieder mit einem LLM erstellen lassen, ich denke, die Aufgabe ist nicht schwer wenn es nur dedizierte Daten erhält. Die Testmenge wird überschaubar bleiben und kann in handarbeit evaluiert werden.

Es soll zwei Datensätze an Testdaten geben. Zum eine Fragen, für deren Beantwortung ein bestimmter Chunk gefunden werden muss, zum anderen solche, deren Antworten über mehrere Chunks verteilt ist.

Für die Singe-Chunk-Fragen werden nur technische Daten verwendet. Um diese filtern zu können muss auf die Metadaten zugegriffen werden, die ein JSON-Objekt mit den Daten enthalten. Eine Möglichkeit ist, den Typen in eine eigene Spalte zu schreiben, ist letztlich ja pro Chunk typisch.

Aus den Datensätzen sollen zufällig Datensätze gezogen und an das LLM zur Generierung der Testfragen gesendet werden.

In [ ]:
import os
import re
import json
import random
from tqdm import tqdm
from mistralai import Mistral
from dotenv import load_dotenv
from collections import defaultdict


load_dotenv()

api_key = os.getenv('MISTRAL_API_KEY')
model = 'mistral-medium-2508'
client = Mistral(api_key=api_key, timeout_ms=120000)

In [ ]:
# Requestfunktion
def agent_request(system_promt, schema, content):

    response = client.chat.complete(
        model = model,
        messages = [
            {
                'role': 'system',
                'content': system_promt
            },
            {
                'role': 'user',
                'content': content,
            }
        ],
        response_format = {
            'type': 'json_object',
            'json_schema': schema,
            'strict': True
        }
    )

    return response

## Dataprep

In [ ]:
# Daten laden
with open('../data/processed/products_chunked.jsonl', 'r', encoding='utf-8') as f:
    products_chunked = [json.loads(line) for line in f]

descs_chunks = [item for item in products_chunked if item.get('metadata', {}).get('chunk_type') == 'desc']
specs_chunks = [item for item in products_chunked if item.get('metadata', {}).get('chunk_type') == 'spec']

print(len(descs_chunks))
print(len(specs_chunks))
    

### Single Chunk Questions

In [ ]:
with open('../data/prompts/test_single_chunk_agent.md', 'r') as f:
    specs_prompt = f.read()

with open('../data/prompts/test_single_chunk_schema.json', 'r') as f:
    specs_schema = json.load(f)

In [ ]:
specs_quests = []
specs_samples = random.sample(specs_chunks, k=25)

# print(specs_samples)
for spec in tqdm(specs_samples, total=len(specs_samples)):

    print("=" * 50)
    print("LLM Response:")

    response = agent_request(specs_prompt, specs_schema, spec['document'])
    # Model sendet gelegentlich ein ``` was raus muss  
    content = response.choices[0].message.content.strip()
    content = re.sub(r'\s*```$', '', content)

    print(response)
    print("=" * 50)
    
    for question in json.loads(content):
        specs_quests.append({
            'product_id': spec['metadata']['product_id'],
            'question': question,
            'answer': spec['document'],
            'chunk_id': spec['id']
        })
    
with open('../data/tests/specs_question.json', 'w', encoding='utf-8') as f:
    json.dump(specs_quests, f, ensure_ascii=False, indent=2)

# print(specs_quests)

### Multi Chunk Questions

In [ ]:
with open('../data/prompts/test_multi_chunk_agent.md', 'r') as f:
    descs_prompt = f.read()

with open('../data/prompts/test_multi_chunk_schema.json', 'r') as f:
    descs_schema = json.load(f)

In [ ]:
# Gruppieren
all_products = defaultdict(list)
for chunk in descs_chunks:
    product_id = chunk.get('metadata', {}).get('product_id')
    all_products[product_id].append(chunk)


# Auswahl an Produkten
selected_products = random.sample(list(all_products.keys()), k=3)

# Quests
descs_quests = []
for product_id in selected_products:
    product_chunks = all_products[product_id]
    descs_samples = []
    
    if len(product_chunks) >= 2:
        num_chunks = min(len(product_chunks), 3)
        sampled_chunks = random.sample(product_chunks, k=num_chunks)
        
        # Nur die document Strings extrahieren
        doc_strings = [c['document'] for c in sampled_chunks]
        descs_samples.append(doc_strings)

    print(descs_samples)
    response = agent_request(descs_prompt, descs_schema, json.dumps(descs_samples))
    
    for question in json.loads(response.choices[0].message.content):
        descs_quests.append({
            'product_id': spec['metadata']['product_id'],
            'question': question,
            'chunk_id': spec['id']
        })
        

with open('../data/tests/multi_question.json', 'w', encoding='utf-8') as f:
    json.dump(descs_quests, f, ensure_ascii=False, indent=2)